# Sinh audio bang model da finetune

**Upload len `MyDrive/f5tts/` truoc khi chay:**

| File | Nguon | Kich thuoc |
|---|---|---|
| `ckpts/model_final.pt` | notebook train da copy sang | 1.35 GB |
| `colab.patch` | `f5tts/` | 3 KB |
| `tts_text.py` | `f5tts/` | 2 KB |
| `vocab.txt` | `f5tts/F5-TTS-Vietnamese/data/your_training_dataset/` | 30 KB |
| `sample_00004.wav` | `.../data/your_training_dataset/wavs/` | 190 KB |

`tts_text.py` giu quy tac tach doan va chuan hoa dau cau — **cung file ma `monitor_server.py` tren Mac dung**, nen audio sinh o hai noi giong nhau. Sua quy tac thi upload lai file nay.

Chon **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/f5tts'
!ls -lh {DRIVE} {DRIVE}/ckpts

In [ ]:
%cd /content
!rm -rf F5-TTS-Vietnamese
!git clone -q https://github.com/nguyenthienhy/F5-TTS-Vietnamese
%cd /content/F5-TTS-Vietnamese
!git checkout -q e74db9d && git apply {DRIVE}/colab.patch
!pip install -q -e . --no-deps
!pip install -q accelerate cached_path click datasets ema_pytorch hydra-core jieba \
    librosa pydub pypinyin safetensors soundfile tomli torchdiffeq transformers \
    vocos x_transformers

In [ ]:
!mkdir -p /content/tts
!cp {DRIVE}/tts_text.py /content/
!cp {DRIVE}/vocab.txt {DRIVE}/sample_00004.wav /content/tts/
!cp {DRIVE}/ckpts/model_final.pt /content/tts/
!ls -lh /content/tts

Nap model **mot lan** cho ca phien. Chay lai cac cell sinh audio ben duoi bao nhieu lan cung duoc ma khong phai nap lai.

In [ ]:
import sys, os
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/F5-TTS-Vietnamese/src')

from importlib.resources import files
from omegaconf import OmegaConf
from f5_tts.infer.utils_infer import (
    cfg_strength, cross_fade_duration, infer_process, load_model, load_vocoder,
    nfe_step, preprocess_ref_audio_text, sway_sampling_coef, target_rms,
)
from f5_tts.model import DiT, UNetT  # noqa: F401

CKPT = '/content/tts/model_final.pt'
VOCAB = '/content/tts/vocab.txt'
REF_AUDIO = '/content/tts/sample_00004.wav'
REF_TEXT = 'tình cờ lạc bước vào một ngôi mộ hoang của quý phi đời trước.'

vocoder = load_vocoder(vocoder_name='vocos')
cfg = OmegaConf.load(str(files('f5_tts').joinpath('configs/F5TTS_Base.yaml'))).model
model = load_model(globals()[cfg.backbone], cfg.arch, CKPT, mel_spec_type='vocos', vocab_file=VOCAB)
ref_audio, ref_text = preprocess_ref_audio_text(REF_AUDIO, REF_TEXT)
print('san sang')

In [ ]:
import time, soundfile as sf
from tts_text import prepare_text, split_text, max_chunk_bytes, join_wavs, SPEED, SENTENCE_GAP

LIMIT = max_chunk_bytes(REF_AUDIO, REF_TEXT)

def tts(text, out_name='audio', speed=SPEED, gap=SENTENCE_GAP):
    parts = split_text(prepare_text(text), LIMIT)
    print(f'{len(parts)} doan, nguong {LIMIT} byte')
    paths, started = [], time.time()
    for i, part in enumerate(parts):
        audio, sr, _ = infer_process(
            ref_audio, ref_text, part, model, vocoder, mel_spec_type='vocos',
            target_rms=target_rms, cross_fade_duration=cross_fade_duration,
            nfe_step=nfe_step, cfg_strength=cfg_strength,
            sway_sampling_coef=sway_sampling_coef, speed=speed, fix_duration=None,
        )
        path = f'/content/tts/_part{i:03d}.wav'
        sf.write(path, audio, sr)
        paths.append(path)
        print(f'  {i+1}/{len(parts)}  {time.time()-started:.0f}s', end='\r')
    wav = f'/content/tts/{out_name}.wav'
    mp3 = f'/content/tts/{out_name}.mp3'
    join_wavs(paths, wav, gap)
    for p in paths:
        os.remove(p)
    !ffmpeg -y -hide_banner -loglevel error -i {wav} {mp3}
    os.remove(wav)
    print(f'\nxong {len(parts)} doan trong {time.time()-started:.0f}s -> {mp3}')
    return mp3

## Sinh audio

Dan van ban vao `TEXT` roi chay. Khong can viet thuong hay bo dau `:` `"` `...` — `prepare_text` tu xu ly.

In [ ]:
from IPython.display import Audio, display

TEXT = """
Phương Ứng Vật liền yên tâm, không phải để bản thân thực sự học lại từ đầu một lượt là được.
Kiểu đó ba bốn tháng thời gian tuyệt đối chẳng đủ dùng, bản thân lại chẳng có được cái tài "nhìn qua là nhớ".
"""

mp3 = tts(TEXT, out_name='chuong01')
display(Audio(mp3))

## Tai ve may

Trinh duyet se hoi cho phep tai nhieu file — bam **Allow**. Neu co nhieu file, nen nen lai roi tai mot lan (cell duoi cung).

In [ ]:
from google.colab import files
import glob

for path in sorted(glob.glob('/content/tts/*.mp3')):
    print(path)
    files.download(path)

In [ ]:
from google.colab import files

!cd /content/tts && zip -q -r /content/audio.zip *.mp3
!ls -lh /content/audio.zip
files.download('/content/audio.zip')